<a href="https://colab.research.google.com/github/vermasachin6102/JoyAI-Echo/blob/main/seed_veo_3__joy_ai_echo_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

# NOTE: deliberately no `import torch` here. Importing torch this early
# caches the base-image's preinstalled version (e.g. 2.11.0) in this
# kernel's memory -- a later `pip install --force-reinstall torch==2.8.0`
# in the setup cell writes the right version to disk, but this already-
# running process would keep returning the cached module on any later
# `import torch`, silently ignoring the pin. Parsing nvidia-smi directly
# avoids importing torch before the pinned install runs.
import subprocess

_result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True, check=True,
)
_name, _mem_mib = [x.strip() for x in _result.stdout.strip().split(",")]
vram_gb = float(_mem_mib) / 1024
print(f"GPU: {_name} | VRAM: {vram_gb:.1f} GB")
if vram_gb < 40:
    print("WARNING: <40GB VRAM — JoyAI-Echo will very likely OOM. Switch to G4 or A100 (Runtime > Change runtime type).")
elif vram_gb < 60:
    print("40GB-class GPU: use the reduced settings in Part 2 (num_frames=121, 480x832).")
else:
    print("Big GPU: you can use full README settings in Part 2 (num_frames=241, 736x1280).")


Thu Jul 23 07:55:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   25C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os, sys, glob, subprocess, shutil
from pathlib import Path

# --- FIX 3: never let the kernel sit in a deleted directory ---
os.chdir("/content")

REPO_ROOT = "/content/JoyAI-Echo-gguf-test"
OUTPUT_DIR = f"{REPO_ROOT}/inference_result"

# Both checkpoints cache to LOCAL disk (/content, 256GB). Drive is a FUSE
# network mount -- reading the 46GB Echo checkpoint through it at load time
# was the actual bottleneck (GPU/RAM idle while it crawled through Drive I/O).
# Local disk costs a fresh download each new Colab runtime instead, but every
# load within the session is then fast local-disk I/O.
LOCAL_HF_CACHE = "/content/hf_cache_local"

os.makedirs(LOCAL_HF_CACHE, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

def run(cmd, cwd=None, label=""):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"--- FAILED: {label or ' '.join(cmd)} (exit {r.returncode}) ---")
        print(r.stdout[-2000:])
        print(r.stderr[-3000:])
        raise RuntimeError(f"Step failed: {label or cmd}")
    return r

# --- HF auth from Colab secret 'hf' ---
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("hf")
print("HF token loaded from Colab secret 'hf'.")

# --- FIX 4: clone only counts if requirements.txt is actually there ---
if not os.path.exists(f"{REPO_ROOT}/requirements.txt"):
    shutil.rmtree(REPO_ROOT, ignore_errors=True)
    print("Cloning repo...")
    run(["git", "clone", "https://github.com/vermasachin6102/JoyAI-Echo.git", REPO_ROOT], label="git clone")
else:
    print("Repo present and complete, skipping clone.")

# --- Testing notebook: always on feature/gemma-gguf-loader, not main ---
print("Checking out feature/gemma-gguf-loader...")
run(["git", "fetch", "origin", "feature/gemma-gguf-loader"], cwd=REPO_ROOT, label="git fetch branch")
run(["git", "checkout", "feature/gemma-gguf-loader"], cwd=REPO_ROOT, label="git checkout branch")
run(["git", "pull", "origin", "feature/gemma-gguf-loader"], cwd=REPO_ROOT, label="git pull branch")
print("Now on branch:", run(["git", "branch", "--show-current"], cwd=REPO_ROOT, label="git branch").stdout.strip())

for sub in ["ltx-core/src", "ltx-pipelines/src", "ltx-distillation/src"]:
    p = os.path.join(REPO_ROOT, sub)
    if p not in sys.path:
        sys.path.insert(0, p)

# --- System deps ---
print("Installing ffmpeg...")
run(["apt-get", "-qq", "update"], label="apt update")
run(["apt-get", "-qq", "install", "-y", "ffmpeg"], label="apt install ffmpeg")

# --- Python deps (pinned CUDA 12.8 stack) ---
print("Installing pinned torch stack (this takes a few minutes)...")
# --force-reinstall is required, not optional: Colab's base image ships a
# newer torch preinstalled, and `pip install torch==2.8.0` without it can
# exit 0 (success) while silently leaving the existing version in place --
# pip's resolver may decide the already-installed version satisfies some
# other preinstalled package's constraint and skip the actual downgrade.
# Confirmed on a genuinely fresh A100 runtime (not a stale-kernel issue).
run(["pip", "install", "--quiet", "--force-reinstall", "--index-url", "https://download.pytorch.org/whl/cu128",
     "torch==2.8.0", "torchvision==0.23.0", "torchaudio==2.8.0"], label="torch install")

# Fail loudly and immediately if the pin didn't stick (e.g. a stale kernel
# session still has an old torch cached from an earlier run in memory --
# `import torch` returns the cached module, not what's freshly on disk).
# Catching this here beats discovering it several steps later as a cryptic
# xformers ABI crash.
import torch as _torch_check
if not _torch_check.__version__.startswith("2.8.0"):
    raise RuntimeError(
        f"torch is {_torch_check.__version__}, expected 2.8.0 -- the pinned "
        "install didn't take effect in this process. Usually means a stale "
        "kernel: Runtime -> Restart session, then rerun all cells from the top."
    )
print(f"torch version confirmed: {_torch_check.__version__}")
del _torch_check
print("Installing requirements.txt...")
run(["pip", "install", "--quiet", "-r", "requirements.txt"], cwd=REPO_ROOT, label="requirements.txt")
print("Installing gguf (for the Gemma GGUF loader test)...")
run(["pip", "install", "--quiet", "gguf"], label="gguf install")
# --- FIX 2: constrained hub install AFTER requirements, never unpinned -U ---
print("Installing huggingface_hub (constrained <1.0 for transformers 4.57.6)...")
run(["pip", "install", "--quiet", "huggingface_hub[cli]>=0.34.0,<1.0"], label="hf hub install")

import torch
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| available:", torch.cuda.is_available())

# --- Checkpoints ---
from huggingface_hub import snapshot_download
os.makedirs(f"{REPO_ROOT}/checkpoints", exist_ok=True)

# Echo checkpoint (~46GB) — LOCAL disk. Resumable if interrupted; re-download
# is the cost of avoiding slow Drive-FUSE reads during every model load.
print("Fetching JoyAI-Echo release checkpoint to local disk...")
echo_dir = snapshot_download(
    repo_id="jdopensource/JoyAI-Echo",
    cache_dir=LOCAL_HF_CACHE,
    allow_patterns=["*.safetensors", "*.json", "*.md"],
)
candidates = glob.glob(os.path.join(echo_dir, "**", "*.safetensors"), recursive=True)
assert candidates, "No .safetensors found in jdopensource/JoyAI-Echo"
dst_echo = f"{REPO_ROOT}/checkpoints/echo-longvideo-release.safetensors"
if os.path.islink(dst_echo) or os.path.exists(dst_echo):
    os.remove(dst_echo)
os.symlink(candidates[0], dst_echo)
print("Echo checkpoint ->", os.path.realpath(dst_echo))

# --- FIX 1: the repo's inference.yaml actually looks for checkpoints/test.safetensors ---
dst_test = f"{REPO_ROOT}/checkpoints/test.safetensors"
if os.path.islink(dst_test) or os.path.exists(dst_test):
    os.remove(dst_test)
os.symlink(dst_echo, dst_test)
print("Config-compat symlink: test.safetensors ->", os.path.realpath(dst_test))

# Gemma (~24GB) — LOCAL disk. Resumable if interrupted.
print("Fetching gemma-3-12b-it to local disk (gated — needs accepted license)...")
gemma_dir = snapshot_download(repo_id="google/gemma-3-12b-it", cache_dir=LOCAL_HF_CACHE)
dst_gemma = f"{REPO_ROOT}/checkpoints/gemma-3-12b"
if os.path.islink(dst_gemma):
    os.remove(dst_gemma)
elif os.path.isdir(dst_gemma):
    shutil.rmtree(dst_gemma)
os.symlink(gemma_dir, dst_gemma)
print("Gemma ->", os.path.realpath(dst_gemma))

# Verify every file the pipeline will open actually resolves
print("\n--- Verification ---")
ok = True
for label, path in [
    ("config's checkpoint (test.safetensors)", dst_test),
    ("gemma shard 5 (the one that kept failing)", f"{dst_gemma}/model-00005-of-00005.safetensors"),
    ("gemma tokenizer", f"{dst_gemma}/tokenizer.model"),
]:
    exists = os.path.exists(path)  # follows symlinks — catches broken links
    print(("OK  " if exists else "MISSING  ") + label)
    ok = ok and exists
print("\nSETUP COMPLETE — go to Part 2." if ok else "\nSetup incomplete — re-run this cell (downloads resume).")

In [ ]:
# --- Test: Gemma3 language-model-only GGUF loader ---
# Verifies: real dequantization, load_state_dict against actual GGUF
# tensor data, no leftover meta params, sane forward-pass output, and
# actual GPU memory used (expect ~7-9GB, vs ~22-24GB for the full bf16
# checkpoint that OOM'd on L4 earlier).
import torch
from huggingface_hub import hf_hub_download

for sub in ["ltx-core/src", "ltx-pipelines/src", "ltx-distillation/src"]:
    p = os.path.join(REPO_ROOT, sub)
    if p not in sys.path:
        sys.path.insert(0, p)

from ltx_distillation.models.text_encoder_wrapper import create_text_encoder_wrapper_from_gguf

print("Downloading GGUF (gated repo, uses HF_TOKEN already set above)...")
gguf_path = hf_hub_download(
    repo_id="google/gemma-3-12b-it-qat-q4_0-gguf",
    filename="gemma-3-12b-it-q4_0.gguf",
)
print("GGUF downloaded ->", gguf_path)

torch.cuda.reset_peak_memory_stats()
wrapper = create_text_encoder_wrapper_from_gguf(
    gguf_path=gguf_path,
    checkpoint_path=dst_test,   # existing JoyAI-Echo checkpoint, for embeddings_processor
    gemma_root=dst_gemma,       # existing gemma dir, for tokenizer/processor files only
    device=torch.device("cuda"),
    dtype=torch.bfloat16,
)

alloc_gb = torch.cuda.memory_allocated() / 1024**3
peak_gb = torch.cuda.max_memory_allocated() / 1024**3
print(f"GPU mem after load: alloc={alloc_gb:.2f}GB peak={peak_gb:.2f}GB")

out = wrapper(["A cat sitting on a red chair."])
vc = out["video_context"]
ac = out["audio_context"]
print("video_context shape:", None if vc is None else tuple(vc.shape))
print("audio_context shape:", None if ac is None else tuple(ac.shape))
print("video_context has NaN:", False if vc is None else torch.isnan(vc).any().item())
print("video_context has Inf:", False if vc is None else torch.isinf(vc).any().item())
print("
TEST PASSED" if vc is not None and not torch.isnan(vc).any() else "
TEST FAILED -- check output above")
